# 02 — Layout Model: Extract Tables & Structure from Policy Documents

**Time**: ~10 minutes · **Model**: `prebuilt-layout` · **No training required**

---

## Insurance Scenario

An underwriter needs to review a **policy schedule document** that contains coverage tables, premium breakdowns, and structured sections. The Layout model extracts:

- **Tables** with rows, columns, and cell content
- **Paragraphs** with semantic roles (title, section heading, footnote)
- **Selection marks** (checkboxes and radio buttons)
- **Text** with bounding box coordinates

---

## Prerequisites

Complete [00-setup-and-basics.ipynb](00-setup-and-basics.ipynb) first.

In [ ]:
import os
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest, AnalyzeResult
import pandas as pd

load_dotenv(dotenv_path=os.path.join("..", ".env"))

_api_key = os.environ.get("DOCUMENTINTELLIGENCE_API_KEY", "")
_credential = AzureKeyCredential(_api_key) if _api_key else DefaultAzureCredential()

client = DocumentIntelligenceClient(
    endpoint=os.environ["DOCUMENTINTELLIGENCE_ENDPOINT"],
    credential=_credential
)
print("✅ Client ready")

## Analyze Document Layout

The Layout model is more powerful than Read — it understands document structure, not just text.

In [ ]:
# Sample document (replace with your policy schedule)
document_url = "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-REST-api-samples/master/curl/form-recognizer/sample-layout.pdf"

poller = client.begin_analyze_document(
    "prebuilt-layout",
    AnalyzeDocumentRequest(url_source=document_url)
)
result: AnalyzeResult = poller.result()

print(f"Pages: {len(result.pages)}")
print(f"Tables: {len(result.tables) if result.tables else 0}")
print(f"Paragraphs: {len(result.paragraphs) if result.paragraphs else 0}")

## Extract Tables

**Why this matters for insurance**: Policy schedules, coverage summaries, and premium breakdowns are almost always in tabular format. The Layout model extracts tables with row/column structure.

In [ ]:
if result.tables:
    for table_idx, table in enumerate(result.tables):
        print(f"\n{'=' * 60}")
        print(f"TABLE {table_idx + 1}: {table.row_count} rows × {table.column_count} columns")
        if table.bounding_regions:
            for region in table.bounding_regions:
                print(f"  Location: Page {region.page_number}")
        print(f"{'=' * 60}")
        
        # Build a 2D array to display as DataFrame
        table_data = {}
        for cell in table.cells:
            row = cell.row_index
            col = cell.column_index
            if row not in table_data:
                table_data[row] = {}
            table_data[row][col] = cell.content
        
        # Convert to DataFrame for clean display
        rows = []
        for row_idx in sorted(table_data.keys()):
            row = []
            for col_idx in range(table.column_count):
                row.append(table_data.get(row_idx, {}).get(col_idx, ""))
            rows.append(row)
        
        if rows:
            # Use first row as header if it looks like a header
            df = pd.DataFrame(rows[1:], columns=rows[0]) if len(rows) > 1 else pd.DataFrame(rows)
            print(df.to_string(index=False))
else:
    print("No tables found in the document.")

## Extract Paragraphs with Semantic Roles

Paragraphs include roles like `title`, `sectionHeading`, `footnote`, `pageHeader`, `pageFooter` — useful for understanding document structure.

In [ ]:
if result.paragraphs:
    # Group paragraphs by role
    roles = {}
    for para in result.paragraphs:
        role = para.role if para.role else "body"
        if role not in roles:
            roles[role] = []
        roles[role].append(para.content[:100])  # First 100 chars
    
    print("Document Structure:")
    print("=" * 60)
    for role, contents in roles.items():
        print(f"\n📌 {role.upper()} ({len(contents)} items):")
        for content in contents[:5]:  # Show first 5 per role
            print(f"   • {content}")
        if len(contents) > 5:
            print(f"   ... and {len(contents) - 5} more")
else:
    print("No paragraphs detected.")

## Detect Selection Marks (Checkboxes)

**Why this matters for insurance**: Claim forms often have checkboxes for incident type, injury type, coverage options, etc.

In [ ]:
total_marks = 0
for page in result.pages:
    if page.selection_marks:
        for mark in page.selection_marks:
            total_marks += 1
            state = "☑" if mark.state == "selected" else "☐"
            print(f"  Page {page.page_number}: {state} {mark.state} (confidence: {mark.confidence:.2%})")

if total_marks == 0:
    print("No selection marks (checkboxes) found in this document.")
    print("Tip: Try with a document that contains checkboxes, like a claim form.")
else:
    print(f"\nTotal selection marks found: {total_marks}")

## Page-Level Details with Line Positions

The Layout model provides bounding polygons for every line — useful for document visualization or highlighting.

In [ ]:
page = result.pages[0]
print(f"Page 1: {page.width} × {page.height} {page.unit}")
print(f"Lines: {len(page.lines) if page.lines else 0}")
print(f"Words: {len(page.words) if page.words else 0}")
print()

# Show first 10 lines with their bounding polygons
if page.lines:
    print("First 10 lines with positions:")
    for i, line in enumerate(page.lines[:10]):
        polygon_str = ", ".join(f"({line.polygon[j]:.0f},{line.polygon[j+1]:.0f})" 
                                for j in range(0, min(len(line.polygon), 8), 2)) if line.polygon else "N/A"
        print(f"  [{i+1:2d}] \"{line.content[:60]}\"")
        print(f"       Position: {polygon_str}")

---

## Key Takeaways

| Feature | Value for Insurance |
|---|---|
| **Table extraction** | Parse coverage tables, premium schedules, benefit summaries |
| **Paragraph roles** | Identify titles, headings, footnotes for document navigation |
| **Selection marks** | Read checkboxes on claim forms (incident type, coverage options) |
| **Bounding boxes** | Highlight or annotate specific regions in documents |

## Next Steps

| Notebook | What You'll Learn |
|---|---|
| [03-prebuilt-invoice.ipynb](03-prebuilt-invoice.ipynb) | Extract structured fields from invoices |
| [07-custom-model.ipynb](07-custom-model.ipynb) | Train a model on your proprietary policy documents |